# GeoReady 02 · Capacity and indicative cost

* **Technical capacity (MW)** = eligible km² × 87 MWdc/km² (fixed-tilt median, Bolinger & Bolinger 2022)
  × a land-use factor (the share of eligible land that could realistically be fenced; base 10%).
* **Annual yield (MWh)** = MW × PVOUT (Global Solar Atlas, which already includes system losses).
* **Connection cost** = per project (line length × route factor × R/km + on-site station and feeder bay),
  times the number of projects the zone could hold.
* **LCOE** = (CAPEX + present value of O&M) ÷ present value of output, real 2025 rand,
  for low, base and high cost cases.

The cost cases bundle optimistic or pessimistic values of every assumption; they are not
probabilities.

In [1]:
import os, sys
from pathlib import Path
def _root():  # the geoready folder: from GEOREADY_ROOT, the working folder, or the QGIS Notebook plugin
    if os.environ.get("GEOREADY_ROOT"):
        return Path(os.environ["GEOREADY_ROOT"])
    starts = [Path.cwd()]
    try:
        from qgis.PyQt.QtCore import QSettings
        starts.append(Path(QSettings().value("QGISNotebook/last_directory", "") or "."))
    except ImportError:
        pass
    return next((p for s in starts for p in [s, *s.parents] if (p / "src" / "geoready").exists()), None)
ROOT = _root()
assert ROOT, "Open this notebook from inside the geoready folder, or set GEOREADY_ROOT"
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
if "qgis" in sys.modules:          # inside QGIS: draw figures off-screen
    import matplotlib; matplotlib.use("Agg")
from geoready.nb import show
CONFIG = "config/northern_cape_pv.yaml"

In [2]:
import pandas as pd
from geoready.run import Run
from geoready import econ
run = Run(CONFIG); cfg = run.cfg
z = pd.read_csv(run.path("zones_stage1.csv"))
cap = cfg["capacity"]
z["capacity_mw"] = econ.capacity_mw(z["eligible_km2"], cap["density_mwdc_per_km2"] * cap["land_use_factor"]["base"])
for case in ("low", "base", "high"):
    r = econ.zone_costs(z["capacity_mw"], z["pvout"], z["grid_km"], cfg["economics"], case)
    z[f"lcoe_{case}"] = r["lcoe"]
    if case == "base":
        z["mwh_per_year"] = r["mwh_per_year"]
        z["connection_share"] = r["connection_capex"] / (r["plant_capex"] + r["connection_capex"])
show(z[["capacity_mw", "mwh_per_year", "connection_share", "lcoe_low", "lcoe_base", "lcoe_high"]].describe().round(2))

,capacity_mw,mwh_per_year,connection_share,lcoe_low,lcoe_base,lcoe_high
count,2132.00,2132.00,2132.00,2132.00,2132.00,2132.00
mean,733.44,1451640.17,0.07,535.46,847.13,1386.30
std,188.46,375777.46,0.03,11.31,24.17,100.98
min,217.50,422965.57,0.02,497.31,771.41,1160.10
25%,640.54,1260223.28,0.05,527.72,830.11,1302.92
50%,841.18,1655227.56,0.07,536.57,846.84,1381.06
75%,868.37,1710613.33,0.10,543.66,865.60,1471.13
max,870.00,1809287.77,0.12,563.89,908.77,1608.13


### Check against recent prices

REIPPPP Bid Window 7 solar PV preferred bidders were priced at about R500–514/MWh (July 2025).
The model's **low** case lands near those bids; the **base** case is well above them, because it
uses fixed-tilt yield (most new plants track the sun), a public-sector discount rate and
conservative CAPEX. Read the cost axis as relative: which zones are cheaper than others is more
robust than the rand level.

In [3]:
band = cfg["economics"]["reference_bids_r_per_mwh"]
show(pd.Series({f"{c} case, median R/MWh": round(z[f"lcoe_{c}"].median()) for c in ("low", "base", "high")}
               | {"BW7 bids, R/MWh": f"{band[0]}-{band[1]}"}))

low case, median R/MWh         537
base case, median R/MWh        847
high case, median R/MWh       1381
BW7 bids, R/MWh            500-514
dtype: object

In [4]:
z.to_csv(run.path("zones_stage2.csv"), index=False)